# UNIVERSAL ADAPTIVE PIPELINE: TEXT CLUSTERING & TOPIC MODELING
### COMPETITION-GRADE ARCHITECTURE (UNSUPERVISED / NO LABELS NEEDED)

**Prinsip Desain:**
1. **Zero Hardcoded Stopwords**: Menggunakan pemfilteran frekuensi korpus adaptif (`max_df` dan `min_df`) yang otomatis mengeliminasi stopwords spesifik domain pada bahasa apa pun (Indonesia, Inggris, atau istilah teknis).
2. **Auto-Schema Discovery**: Otomatis mendeteksi kolom ID dan kolom Teks utama tanpa asumsi nama kolom.
3. **Multi-Algorithm Clustering**: K-Means, MiniBatchKMeans, dan Agglomerative (Hierarchical) Clustering.
4. **Automated K-Selection**: Evaluasi kurva **Elbow Method (Inertia)** dan **Silhouette Score** secara otomatis untuk rentang klaster yang fleksibel.
5. **c-TF-IDF Topic Keyword Extraction**: Mengekstrak kata kunci representatif per klaster secara matematis untuk memudahkan penamaan topik.
6. **Format Submission Standar**: Menghasilkan berkas CSV berisi `[ID_COL], Cluster` yang siap diuji dengan kriteria *Adjusted Rand Index (ARI)*.


In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

SEED = 2026
print("Environment Clustering & Topic Modeling siap. Seed:", SEED)


## 1. Unified Configuration Card


In [ ]:
CONFIG = {
    # Jalur dataset teks tanpa label (CSV, TSV, Parquet, XLSX)
    "dataset_path": "web_mining_news.csv",
    
    # Kolom: Set None untuk auto-detect, atau masukkan string nama kolom
    "id_col": None,
    "text_cols": None, # Kolom yang akan diklasterkan (bisa single string atau list)
    
    # Rentang pencarian jumlah klaster optimal (K)
    "min_k": 2,
    "max_k": 10,
    
    # Algoritma klaster yang digunakan: 'kmeans', 'minibatch', atau 'agglomerative'
    "algorithm": "kmeans",
    
    # Preprocessing
    "lower": True,
    "strip_urls": True,
    "strip_html": True,
    "max_features": 5000,
    "lsa_components": 30 # Komponen reduksi dimensi LSA (TruncatedSVD)
}

print("Konfigurasi tersimpan.")


## 2. Auto-Discovery & Data Ingestion Engine


In [ ]:
def load_unsupervised_data(cfg):
    p = cfg["dataset_path"]
    if not os.path.exists(p):
        print(f"Peringatan: Berkas '{p}' belum ada. Menjalankan fallback generator berita trending...")
        df = pd.DataFrame({
            "news_id": [f"ID_{i:03d}" for i in range(1, 21)],
            "headline": [
                "Timnas U23 Indonesia Lolos Semifinal Piala Asia Usai Kalahkan Korea",
                "Jadwal Lengkap Pertandingan Timnas Indonesia di Piala Asia U23 Qatar",
                "Shin Tae-yong Puji Mental Pemain Timnas Usai Laga Sengit Adu Penalti",
                "Profil Wasit Laga Timnas Indonesia vs Korea Selatan di Perempat Final",
                "Puncak Arus Mudik Lebaran 2024 Jalur Tol Trans Jawa Mulai Padat",
                "Jadwal Lengkap Rekayasa One Way dan Contraflow Tol Trans Jawa",
                "Kakorlantas Imbau Pemudik Tidak Istirahat di Bahu Jalan Tol",
                "Pantauan Arus Balik H+3 Lebaran Pelabuhan Bakauheni Mulai Ramai",
                "Rupiah Melemah Terhadap Dollar AS Dampak Konflik Geopolitik Global",
                "IHSG Diprediksi Menguat Pasca Libur Lebaran Cek Saham Pilihan",
                "Harga Emas Antam Naik Tajam Hari Ini Simak Rincian Terbarunya",
                "Bank Indonesia Siapkan Langkah Intervensi Stabilkan Nilai Tukar Rupiah",
                "Apple Rilis Pembaruan iOS Terbaru untuk Atasi Bug Keamanan",
                "Perkembangan AI Generatif Berpotensi Ubah Lanskap Industri Teknologi",
                "Spesifikasi dan Harga Smartphone Terbaru yang Rilis Bulan Ini",
                "Google Uji Fitur Pencarian Berbasis Kecerdasan Buatan Generative AI",
                "BMKG Keluarkan Peringatan Dini Cuaca Ekstrem Hujan Lebat di Wilayah Ini",
                "Dampak Gempa Bumi Magnitudo 6.2 di Wilayah Pesisir Barat Sumatera",
                "Prakiraan Cuaca Jabodetabek Hari Ini Berpotensi Hujan Disertai Angin",
                "Status Aktivitas Gunung Berapi Meningkat Warga Diminta Jauhi Zona Bahaya"
            ],
            "article_url": [f"https://berita.com/read/2024/{i}" for i in range(1, 21)]
        })
    else:
        ext = os.path.splitext(p)[1].lower()
        if ext in ['.xlsx', '.xls']: df = pd.read_excel(p)
        elif ext == '.parquet': df = pd.read_parquet(p)
        else: df = pd.read_csv(p)

    # Auto-detect ID Col
    id_col = cfg["id_col"]
    if not id_col or id_col not in df.columns:
        id_cands = [c for c in df.columns if re.search(r"id|uuid|kode|no", c, re.IGNORECASE)]
        id_col = id_cands[0] if id_cands else df.columns[0]

    # Auto-detect Text Col
    text_cols = cfg["text_cols"]
    if not text_cols:
        text_cands = [c for c in df.columns if c != id_col and (df[c].dtype == object or pd.api.types.is_string_dtype(df[c]))]
        if not text_cands: text_cands = [df.columns[1]]
        text_cols = text_cands
    elif isinstance(text_cols, str):
        text_cols = [text_cols]

    return df, id_col, text_cols

df, ID_COL, TEXT_COLS = load_unsupervised_data(CONFIG)
print(f"=== HASIL DETEKSI SKEMA CLUSTERING ===")
print(f"Kolom ID      : '{ID_COL}'")
print(f"Kolom Teks    : {TEXT_COLS}")
print(f"Total Dokumen : {len(df)}")
print(df.head(3))


## 3. Domain-Agnostic Text Cleaning & Corpus Filtering
Menggunakan parameter `max_df` dan `min_df` alih-alih daftar stopword kaku, sehingga kata-kata yang terlalu umum (muncul >85% dokumen) otomatis dibuang pada bahasa apa pun.


In [ ]:
def clean_cluster_text(s, lower=True, strip_urls=True, strip_html=True):
    if not isinstance(s, str): return ""
    if strip_html: s = re.sub(r"<[^>]+>", " ", s)
    if strip_urls: s = re.sub(r"https?://\S+|www\.\S+", " ", s)
    s = re.sub(r"(.)\1{2,}", r"\1", s) # normalisasi huruf repetitif
    s = re.sub(r"[^\w\s]", " ", s)
    if lower: s = s.lower()
    return re.sub(r"\s+", " ", s).strip()

# Gabungkan kolom teks jika >1
cleaned_series = []
for c in TEXT_COLS:
    cleaned_series.append(df[c].fillna("").astype(str).apply(lambda x: clean_cluster_text(x, CONFIG["lower"], CONFIG["strip_urls"], CONFIG["strip_html"])))

if len(cleaned_series) == 1:
    df["processed_text"] = cleaned_series[0]
else:
    fused = cleaned_series[0]
    for s in cleaned_series[1:]: fused = fused + " " + s
    df["processed_text"] = fused.str.strip()

print("Contoh Teks Siap Klasterisasi:")
for i in range(min(3, len(df))):
    print(f"[{i}] {df['processed_text'].iloc[i]}")


## 4. Ekstraksi Vektor TF-IDF & Reduksi Dimensi LSA


In [ ]:
# max_df=0.85 membuang kata yang muncul di >85% dokumen (auto-stopwords)
# min_df=2 membuang typo unik yang hanya muncul 1 kali
min_df_val = 2 if len(df) >= 20 else 1
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=CONFIG["max_features"],
    sublinear_tf=True,
    max_df=0.85,
    min_df=min_df_val
)
X_tfidf = tfidf.fit_transform(df["processed_text"])

# Reduksi Dimensi LSA untuk menyaring komponen semantik utama
n_lsa = min(CONFIG["lsa_components"], X_tfidf.shape[1] - 1, len(df) - 1)
n_lsa = max(2, n_lsa)
svd = TruncatedSVD(n_components=n_lsa, random_state=SEED)
X_dense = svd.fit_transform(X_tfidf)

print(f"Matriks TF-IDF Asli : {X_tfidf.shape}")
print(f"Matriks LSA Tereduksi: {X_dense.shape} (Siap Klasterisasi)")


## 5. Automated K-Selection (Elbow Method & Silhouette Score)


In [ ]:
max_search_k = min(CONFIG["max_k"], len(df) - 1)
min_search_k = min(CONFIG["min_k"], max_search_k - 1)
min_search_k = max(2, min_search_k)

k_range = list(range(min_search_k, max_search_k + 1))
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_dense)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_dense, labels)
    sil_scores.append(sil)
    print(f"K = {k:2d} | Silhouette Score: {sil:.4f} | Inertia: {km.inertia_:.2f}")

# Plot Kurva Diagnostik
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(k_range, inertias, 'bo-', label="Inertia (Elbow)")
ax1.set_xlabel("Jumlah Klaster (K)")
ax1.set_ylabel("Inertia", color='b')

ax2 = ax1.twinx()
ax2.plot(k_range, sil_scores, 'ro--', label="Silhouette Score")
ax2.set_ylabel("Silhouette Score", color='r')
plt.title("Analisis Optimasi Jumlah Klaster (Elbow vs Silhouette Index)")
plt.grid(True, alpha=0.3)
plt.show()

best_k = k_range[np.argmax(sil_scores)]
print(f"\n>>> Rekomendasi K Terbaik: K = {best_k} (Silhouette Score: {max(sil_scores):.4f}) <<<")


## 6. Eksekusi Klasterisasi Akhir


In [ ]:
CHOSEN_K = best_k # Anda dapat mengganti nilai ini secara manual jika menginginkan K tertentu

algo = CONFIG.get("algorithm", "kmeans").lower()
if algo == "minibatch":
    clusterer = MiniBatchKMeans(n_clusters=CHOSEN_K, random_state=SEED, batch_size=256)
elif algo == "agglomerative":
    clusterer = AgglomerativeClustering(n_clusters=CHOSEN_K)
else:
    clusterer = KMeans(n_clusters=CHOSEN_K, random_state=SEED, n_init=20)

df["Cluster"] = clusterer.fit_predict(X_dense)
print(f"Klasterisasi Selesai dengan {algo.upper()} (K={CHOSEN_K}).")
print("Distribusi Anggota per Klaster:")
print(df["Cluster"].value_counts().sort_index())


## 7. Interpretasi Topik: Kata Kunci Dominan per Klaster (c-TF-IDF)


In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
print("=== INTERPRETASI KATA KUNCI TOPIK SETIAP KLASTER ===")

for c in sorted(df["Cluster"].unique()):
    c_docs = df[df["Cluster"] == c]
    if len(c_docs) == 0: continue
    
    # Class-based TF-IDF aggregation
    c_text = " ".join(c_docs["processed_text"])
    c_vec = tfidf.transform([c_text]).toarray()[0]
    top_kw_idx = np.argsort(c_vec)[-8:][::-1]
    top_keywords = feature_names[top_kw_idx]
    
    print(f"\n[Klaster {c}] (Total: {len(c_docs)} Dokumen)")
    print("Kata Kunci Dominan:", ", ".join(top_keywords))
    print("Contoh Teks:")
    for sample in c_docs[TEXT_COLS[0]].head(2):
        print(f"  * {sample}")


## 8. Ekspor Berkas Submission Format `ID, Cluster`


In [ ]:
sub_df = pd.DataFrame({
    ID_COL: df[ID_COL],
    "Cluster": df["Cluster"]
})
out_name = "submission_unsupervised_clusters.csv"
sub_df.to_csv(out_name, index=False)
print(f"Berkas submission '{out_name}' berhasil disimpan:")
print(sub_df.head(10))
